# ToolTune Training Pipeline
**Training better tool-using agents with SFT + GRPO**

This notebook runs the complete ToolTune training pipeline on a T4 GPU:
1. Clone repo & install deps
2. Generate 500 tasks across 5 tools
3. SFT warmup (teaches format) ~30 min
4. GRPO-Balanced (teaches behavior) ~2.5 hrs
5. Generate eval traces for Base vs SFT vs GRPO
6. Run evaluation & show comparison table

**Runtime:** ~4-5 hours total on T4

**Important:** Go to Runtime > Change runtime type > Select **T4 GPU**

In [ ]:
# Verify GPU
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > T4 GPU"

## 1. Setup: Clone Repo & Install Dependencies

In [ ]:
# Clone the repo
!git clone https://github.com/harneet2512/Codetune.git 2>/dev/null || (cd Codetune && git pull)
%cd Codetune

# Install dependencies
!pip install -q -e ".[all]" 2>&1 | tail -3
!pip install -q bitsandbytes accelerate peft trl datasets transformers sentencepiece protobuf 2>&1 | tail -3

print("\n=== Setup complete ===")

## 2. Generate 500 Tasks Across All 5 Tools

In [ ]:
!python -m tasks.generate_tasks

In [ ]:
# Quick sanity check
import json

for f in ['tasks/tier1_single_tool.json', 'tasks/tier2_restraint.json',
          'tasks/tier3_multi_step.json', 'tasks/tier4_error_recovery.json']:
    tasks = json.load(open(f))
    print(f"{f}: {len(tasks)} tasks")

# Show sample from each tier
t1 = json.load(open('tasks/tier1_single_tool.json'))
print(f"\nSample Tier 1: {t1[0]['prompt']}")
print(f"  -> {t1[0]['ground_truth']} (tools: {t1[0]['expected_tools']})")

t3 = json.load(open('tasks/tier3_multi_step.json'))
print(f"\nSample Tier 3: {t3[0]['prompt']}")
print(f"  -> {t3[0]['ground_truth']} (tools: {t3[0]['expected_tools']})")

## 3. SFT Warmup (~30 min)
Teaches the model the `<think>`, `<tool_call>`, `<observation>`, `<answer>` format.
This is supervised fine-tuning on synthetic gold traces.

In [ ]:
!python -m train.sft_tooltune \
    --base-model Qwen/Qwen2.5-7B-Instruct \
    --output-dir outputs/tooltune-sft-checkpoints \
    --task-files \
        tasks/tier1_single_tool.json \
        tasks/tier2_restraint.json \
        tasks/tier3_multi_step.json

In [ ]:
# Verify SFT output exists
import os
adapter_path = 'outputs/tooltune-sft-checkpoints/final_adapter'
if os.path.exists(adapter_path):
    files = os.listdir(adapter_path)
    print(f"SFT adapter saved: {len(files)} files")
    print(files)
else:
    print("ERROR: SFT adapter not found!")
    print("Available:", os.listdir('outputs/tooltune-sft-checkpoints') if os.path.exists('outputs/tooltune-sft-checkpoints') else 'nothing')

## 4. GRPO-Balanced Training (~2.5 hrs)
This is the main RL training. The model learns:
- **When** to use tools (restraint on trivial tasks)
- **Which** tool to pick (correct selection)
- **How** to chain multi-step operations
- **How** to recover from tool errors

All from binary task completion reward + subsidiary signals.

In [ ]:
!python -m train.grpo_tooltune \
    --base-model outputs/tooltune-sft-checkpoints/final_adapter \
    --variant grpo-balanced \
    --task-files \
        tasks/tier1_single_tool.json \
        tasks/tier2_restraint.json \
        tasks/tier3_multi_step.json \
        tasks/tier4_error_recovery.json

In [ ]:
# Verify GRPO output
grpo_path = 'outputs/tooltune-grpo-balanced'
if os.path.exists(grpo_path):
    files = os.listdir(grpo_path)
    print(f"GRPO model saved: {len(files)} files")
    print(files)
else:
    print("ERROR: GRPO output not found!")

## 5. Generate Eval Traces
Load each model variant and run real agentic loops (with actual tool execution)
to produce traces for evaluation.

In [ ]:
# Generate traces for base model
!python -m train.generate_traces \
    --output-dir results/traces \
    --variants base

In [ ]:
# Generate traces for SFT model
!python -m train.generate_traces \
    --output-dir results/traces \
    --variants sft

In [ ]:
# Generate traces for GRPO-Balanced model
!python -m train.generate_traces \
    --output-dir results/traces \
    --variants grpo-balanced

## 6. Evaluation: The Money Shot
Run all 4 eval suites and produce the cross-variant comparison table.

In [ ]:
import json
from pathlib import Path

# Run evals for each variant
results = {}
for variant in ['base', 'sft', 'grpo-balanced']:
    trace_file = f'results/traces/{variant}.json'
    output_file = f'results/eval/{variant}.json'
    if Path(trace_file).exists():
        Path('results/eval').mkdir(parents=True, exist_ok=True)
        !python -m eval.run_all --input {trace_file} --output {output_file}
        results[variant] = json.load(open(output_file))
    else:
        print(f"Skipping {variant}: no traces found")

In [ ]:
# Build the cross-variant comparison table
print("=" * 80)
print("  TOOLTUNE CROSS-VARIANT EVALUATION")
print("=" * 80)
print()

header = f"{'Metric':<35} {'Base (V0)':>12} {'SFT (V1)':>12} {'GRPO (V4)':>12}"
print(header)
print("-" * len(header))

# Collect all metrics across suites
all_metrics = set()
for v, data in results.items():
    for suite_name, suite_data in data.get('suites', {}).items():
        for metric in suite_data.get('metrics', {}):
            all_metrics.add(f"{suite_name}/{metric}")

for metric_path in sorted(all_metrics):
    suite, metric = metric_path.split('/', 1)
    values = []
    for variant in ['base', 'sft', 'grpo-balanced']:
        if variant in results:
            val = results[variant].get('suites', {}).get(suite, {}).get('metrics', {}).get(metric)
            if val is not None:
                if isinstance(val, float):
                    values.append(f"{val:.4f}")
                else:
                    values.append(str(val))
            else:
                values.append('--')
        else:
            values.append('--')
    
    short_name = metric.replace('_', ' ').title()
    print(f"{short_name:<35} {values[0]:>12} {values[1]:>12} {values[2]:>12}")

print()
print("=" * 80)

## 7. Showcase: Side-by-Side Trace Comparison
Show the 6 flagship examples where the behavioral difference is most dramatic.

In [ ]:
import json
from pathlib import Path

showcase = json.load(open('tooltune_data/showcase_examples.json'))

# Load traces
traces = {}
for variant in ['base', 'grpo-balanced']:
    tf = Path(f'results/traces/{variant}.json')
    if tf.exists():
        all_traces = json.load(open(tf))
        traces[variant] = {t['task']['prompt']: t for t in all_traces}

for example in showcase:
    print("=" * 70)
    print(f"  {example['title']}")
    print(f"  Failure mode: {example['failure_mode']}")
    print("=" * 70)
    print(f"Task: {example['task']}")
    print()

    for variant, label in [('base', 'BASE (V0)'), ('grpo-balanced', 'GRPO-BALANCED (V4)')]:
        if variant in traces and example['task'] in traces[variant]:
            trace = traces[variant][example['task']]
            print(f"--- {label} ---")
            print(f"Final answer: {trace.get('final_answer', 'N/A')[:100]}")
            print(f"Tool calls: {[tc['name'] for tc in trace.get('tool_calls', [])]}")
            print(f"Transcript (first 500 chars):")
            print(trace.get('transcript', 'N/A')[:500])
        else:
            print(f"--- {label} ---")
            expected_key = f'expected_{variant.split("-")[0]}'
            print(f"  (no trace) Expected: {example.get(expected_key, 'N/A')}")
        print()
    print()

## 8. Save Results to Google Drive
Save trained models and eval results so they persist after Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create output directory in Drive
!mkdir -p /content/drive/MyDrive/tooltune_results

# Copy trained models
!cp -r outputs/tooltune-sft-checkpoints /content/drive/MyDrive/tooltune_results/ 2>/dev/null || echo 'SFT not found'
!cp -r outputs/tooltune-grpo-balanced /content/drive/MyDrive/tooltune_results/ 2>/dev/null || echo 'GRPO not found'

# Copy traces and eval results
!cp -r results/ /content/drive/MyDrive/tooltune_results/

print("\nSaved to Google Drive: /MyDrive/tooltune_results/")
!ls -la /content/drive/MyDrive/tooltune_results/

## 9. (Optional) Download Models Locally
Download the trained adapters to use in the playground.

In [ ]:
# Zip for download
!cd outputs && tar czf /content/tooltune-models.tar.gz tooltune-sft-checkpoints/ tooltune-grpo-balanced/ 2>/dev/null
!cd results && tar czf /content/tooltune-results.tar.gz traces/ eval/ 2>/dev/null

from google.colab import files
# Uncomment to download:
# files.download('/content/tooltune-models.tar.gz')
# files.download('/content/tooltune-results.tar.gz')
print("Uncomment the lines above to download, or use the Drive copy.")